In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter


In [2]:
# --- 1. 超参数配置 ---
hparams = {
    'dataset': 'car',
    'hidden_channels': 32,
    'heads': 4,
    'topk_values': [8, 32],
    'cpe_profile_bins': 8,
    'learning_rate': 0.005,
    'weight_decay': 5e-4,
    'epochs': 150,
    'dropout': 0.5,
    'seed': 42
}

torch.manual_seed(hparams['seed'])
np.random.seed(hparams['seed'])


In [3]:
# --- 2. 数据读取工具函数 ---
def load_labels(base_path, dataset_name, expected_num_nodes):
    candidate_paths = [
        f"{base_path}{dataset_name}.data",
        f"{base_path}{dataset_name}.data.csv",
    ]

    for labels_path in candidate_paths:
        try:
            df = pd.read_csv(labels_path, header=None)
        except FileNotFoundError:
            continue

        df = df.dropna(axis=1, how='all')
        if len(df) != expected_num_nodes:
            continue
        return df.iloc[:, -1].values

    raise ValueError(f"无法读取与对象数量 {expected_num_nodes} 匹配的标签文件: {candidate_paths}")


def load_feature_matrix(path):
    values = np.loadtxt(path, delimiter=',')
    if values.ndim == 1:
        values = values.reshape(1, -1)
    return torch.tensor(values, dtype=torch.float)


def load_depth_profile_cpe(base_path, dataset_name, profile_bins):
    cpe_pos = load_feature_matrix(f"{base_path}{dataset_name}_CPE_A_plus_depth_profile{profile_bins}.csv")
    cpe_neg = load_feature_matrix(f"{base_path}{dataset_name}_CPE_A_negative_depth_profile{profile_bins}.csv")
    return cpe_pos, cpe_neg


def keep_topk_memberships_per_object(df, topk):
    if topk is None or topk <= 0 or len(df) == 0:
        return df

    return (df.sort_values(['object_id', 'weight', 'concept_id'], ascending=[True, False, True])
              .groupby('object_id', group_keys=False)
              .head(topk)
              .reset_index(drop=True))


def load_bipartite_edges(path, object_count, concept_count, topk_per_object):
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        gz_path = path + '.gz'
        df = pd.read_csv(gz_path, compression='gzip')
    required_columns = {'object_id', 'concept_id', 'weight'}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"边表必须包含列 {required_columns}: {path}")

    original_edge_count = len(df)
    df = keep_topk_memberships_per_object(df, topk_per_object)

    object_ids = torch.tensor(df['object_id'].to_numpy(), dtype=torch.long)
    concept_ids = torch.tensor(df['concept_id'].to_numpy(), dtype=torch.long)
    weights = torch.tensor(df['weight'].to_numpy(), dtype=torch.float).view(-1, 1)

    if object_ids.numel() > 0:
        if object_ids.min() < 0 or object_ids.max() >= object_count:
            raise ValueError(f"对象 id 超出范围: {path}")
        if concept_ids.min() < 0 or concept_ids.max() >= concept_count:
            raise ValueError(f"概念 id 超出范围: {path}")

    obj_to_concept = torch.stack([object_ids, concept_ids], dim=0)
    concept_to_obj = torch.stack([concept_ids, object_ids], dim=0)
    return {
        'obj_to_concept': obj_to_concept,
        'concept_to_obj': concept_to_obj,
        'edge_attr': weights,
        'rev_edge_attr': weights.clone(),
        'original_edge_count': original_edge_count,
        'kept_edge_count': len(df),
    }


In [4]:
# --- 3. 构建二部图张量包 ---
def load_bipartite_tensors(dataset_name, topk_memberships_per_object, seed, cpe_profile_bins):
    base_path = f'../data/{dataset_name}/'

    x_raw = load_feature_matrix(f"{base_path}{dataset_name}.data.cleaned.csv")
    num_objects = x_raw.shape[0]
    cpe_pos, cpe_neg = load_depth_profile_cpe(base_path, dataset_name, cpe_profile_bins)
    if cpe_pos.shape[0] != num_objects or cpe_neg.shape[0] != num_objects:
        raise ValueError(
            f"CPE 行数必须和对象数量一致: num_objects={num_objects}, "
            f"cpe_pos={cpe_pos.shape[0]}, cpe_neg={cpe_neg.shape[0]}"
        )
    x_pos = torch.cat([x_raw, cpe_pos], dim=1)
    x_neg = torch.cat([x_raw, cpe_neg], dim=1)

    pos_concept_x = load_feature_matrix(f"{base_path}{dataset_name}_positive_object_concept_concept_features.csv")
    neg_concept_x = load_feature_matrix(f"{base_path}{dataset_name}_negative_object_concept_concept_features.csv")

    pos_edges = load_bipartite_edges(
        f"{base_path}{dataset_name}_positive_object_concept_edges.csv",
        num_objects,
        pos_concept_x.shape[0],
        topk_memberships_per_object
    )
    neg_edges = load_bipartite_edges(
        f"{base_path}{dataset_name}_negative_object_concept_edges.csv",
        num_objects,
        neg_concept_x.shape[0],
        topk_memberships_per_object
    )

    labels_numpy = load_labels(base_path, dataset_name, num_objects)
    encoder = LabelEncoder()
    y_numpy = encoder.fit_transform(labels_numpy)
    y = torch.tensor(y_numpy, dtype=torch.long)

    generator = torch.Generator().manual_seed(seed)
    num_train = int(num_objects * 0.6)
    num_val = int(num_objects * 0.2)
    indices = torch.randperm(num_objects, generator=generator)
    train_mask = torch.zeros(num_objects, dtype=torch.bool); train_mask[indices[:num_train]] = True
    val_mask = torch.zeros(num_objects, dtype=torch.bool); val_mask[indices[num_train:num_train + num_val]] = True
    test_mask = torch.zeros(num_objects, dtype=torch.bool); test_mask[indices[num_train + num_val:]] = True

    print(f"topK={topk_memberships_per_object}")
    print(f"对象原始特征维度: {x_raw.shape[1]}")
    print(f"正概念 profile{cpe_profile_bins} CPE 维度: {cpe_pos.shape[1]}")
    print(f"负概念 profile{cpe_profile_bins} CPE 维度: {cpe_neg.shape[1]}")
    print(f"正分支对象特征维度: {x_pos.shape[1]}")
    print(f"负分支对象特征维度: {x_neg.shape[1]}")
    print(f"正概念节点数: {pos_concept_x.shape[0]}, 正概念特征维度: {pos_concept_x.shape[1]}, 正边数: {pos_edges['kept_edge_count']}/{pos_edges['original_edge_count']}")
    print(f"负概念节点数: {neg_concept_x.shape[0]}, 负概念特征维度: {neg_concept_x.shape[1]}, 负边数: {neg_edges['kept_edge_count']}/{neg_edges['original_edge_count']}")

    return {
        'x_pos': x_pos,
        'x_neg': x_neg,
        'pos_concept_x': pos_concept_x,
        'neg_concept_x': neg_concept_x,
        'pos_edges': pos_edges,
        'neg_edges': neg_edges,
        'y': y,
        'train_mask': train_mask,
        'val_mask': val_mask,
        'test_mask': test_mask,
        'num_classes': len(np.unique(y_numpy)),
    }


In [5]:
# --- 4. 定义真正使用 edge_attr 的二部图 Transformer ---
class WeightedBipartiteBranch(nn.Module):
    def __init__(self, object_in_channels, concept_in_channels, hidden_channels, heads=4, dropout=0.5):
        super(WeightedBipartiteBranch, self).__init__()
        self.dropout = dropout
        self.object_encoder = nn.Linear(object_in_channels, hidden_channels)
        self.concept_encoder = nn.Linear(concept_in_channels, hidden_channels)

        # 两个方向都使用 edge_dim=1，因此 membership weight 会进入注意力计算。
        self.object_to_concept = TransformerConv(
            hidden_channels,
            hidden_channels,
            heads=heads,
            edge_dim=1,
            concat=False
        )
        self.concept_to_object = TransformerConv(
            hidden_channels,
            hidden_channels,
            heads=heads,
            edge_dim=1,
            concat=False
        )

    def forward(self, object_x, concept_x, obj_to_concept, concept_to_obj, edge_attr, rev_edge_attr):
        object_h0 = self.object_encoder(object_x)
        concept_h0 = self.concept_encoder(concept_x)

        concept_h = self.object_to_concept(
            (object_h0, concept_h0),
            obj_to_concept,
            edge_attr
        )
        concept_h = F.dropout(F.relu(concept_h), p=self.dropout, training=self.training)

        object_msg = self.concept_to_object(
            (concept_h, object_h0),
            concept_to_obj,
            rev_edge_attr
        )
        object_msg = F.dropout(F.relu(object_msg), p=self.dropout, training=self.training)

        # 保留对象自身编码，避免二部图消息过强时覆盖原始特征。
        return object_h0 + object_msg


class DualWeightedBipartiteTransformer(nn.Module):
    def __init__(self, pos_object_in_channels, neg_object_in_channels, pos_concept_channels, neg_concept_channels,
                 hidden_channels, out_channels, heads=4, dropout=0.5):
        super(DualWeightedBipartiteTransformer, self).__init__()
        self.pos_branch = WeightedBipartiteBranch(
            pos_object_in_channels,
            pos_concept_channels,
            hidden_channels,
            heads=heads,
            dropout=dropout
        )
        self.neg_branch = WeightedBipartiteBranch(
            neg_object_in_channels,
            neg_concept_channels,
            hidden_channels,
            heads=heads,
            dropout=dropout
        )
        self.fusion_layer = nn.Linear(hidden_channels * 2, out_channels)

    def forward(self, batch):
        pos_h = self.pos_branch(
            batch['x_pos'],
            batch['pos_concept_x'],
            batch['pos_edges']['obj_to_concept'],
            batch['pos_edges']['concept_to_obj'],
            batch['pos_edges']['edge_attr'],
            batch['pos_edges']['rev_edge_attr'],
        )
        neg_h = self.neg_branch(
            batch['x_neg'],
            batch['neg_concept_x'],
            batch['neg_edges']['obj_to_concept'],
            batch['neg_edges']['concept_to_obj'],
            batch['neg_edges']['edge_attr'],
            batch['neg_edges']['rev_edge_attr'],
        )
        return self.fusion_layer(torch.cat([pos_h, neg_h], dim=1))


In [6]:
# --- 5. 单组 topK 实验 ---
def run_experiment(topk):
    torch.manual_seed(hparams['seed'])
    np.random.seed(hparams['seed'])

    batch = load_bipartite_tensors(hparams['dataset'], topk, hparams['seed'], hparams['cpe_profile_bins'])
    model = DualWeightedBipartiteTransformer(
        pos_object_in_channels=batch['x_pos'].shape[1],
        neg_object_in_channels=batch['x_neg'].shape[1],
        pos_concept_channels=batch['pos_concept_x'].shape[1],
        neg_concept_channels=batch['neg_concept_x'].shape[1],
        hidden_channels=hparams['hidden_channels'],
        out_channels=batch['num_classes'],
        heads=hparams['heads'],
        dropout=hparams['dropout']
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=hparams['learning_rate'], weight_decay=hparams['weight_decay'])
    criterion = torch.nn.CrossEntropyLoss()

    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    log_dir_name = f"../runs/{hparams['dataset']}_weighted_bipartite_transformer_cpe_profile{hparams['cpe_profile_bins']}_topk{topk}_{timestamp}"
    writer = SummaryWriter(log_dir_name)
    print(f"TensorBoard 日志将保存在: {log_dir_name}")

    def train(epoch):
        model.train()
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out[batch['train_mask']], batch['y'][batch['train_mask']])
        loss.backward()
        optimizer.step()
        writer.add_scalar('Loss/train', loss.item(), epoch)
        return loss.item()

    def evaluate(epoch):
        model.eval()
        with torch.no_grad():
            out = model(batch)
            pred = out.argmax(dim=1)
            train_acc = (pred[batch['train_mask']] == batch['y'][batch['train_mask']]).sum().item() / batch['train_mask'].sum().item()
            val_acc = (pred[batch['val_mask']] == batch['y'][batch['val_mask']]).sum().item() / batch['val_mask'].sum().item()
            test_acc = (pred[batch['test_mask']] == batch['y'][batch['test_mask']]).sum().item() / batch['test_mask'].sum().item()
            writer.add_scalar('Accuracy/train', train_acc, epoch)
            writer.add_scalar('Accuracy/validation', val_acc, epoch)
            writer.add_scalar('Accuracy/test', test_acc, epoch)
            return train_acc, val_acc, test_acc

    print()
    print(f"--- 开始训练 weighted bipartite Transformer, topK={topk} ---")
    for epoch in range(1, hparams['epochs'] + 1):
        loss = train(epoch)
        train_acc, val_acc, test_acc = evaluate(epoch)
        print(f'topK={topk}, Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

    final_train_acc, final_val_acc, final_test_acc = evaluate(hparams['epochs'])
    metrics = {
        'accuracy/final_train': final_train_acc,
        'accuracy/final_validation': final_val_acc,
        'accuracy/final_test': final_test_acc,
    }
    hparams_for_log = {k: v for k, v in hparams.items() if isinstance(v, (int, float, str, bool))}
    hparams_for_log['topk'] = topk
    writer.add_hparams(hparams_for_log, metrics)
    writer.close()

    print(f"--- topK={topk} 训练完成 ---")
    print(f"topK={topk} 最终测试集准确率: {final_test_acc:.4f}")
    return {
        'topk': topk,
        'final_train_acc': final_train_acc,
        'final_val_acc': final_val_acc,
        'final_test_acc': final_test_acc,
        'log_dir': log_dir_name,
    }


In [7]:
# --- 6. 依次运行 topK=8 和 topK=32 ---
results = []
for topk in hparams['topk_values']:
    results.append(run_experiment(topk))

print()
print("--- 实验汇总 ---")
for result in results:
    print(result)


topK=8
对象原始特征维度: 25
正概念 profile8 CPE 维度: 9
负概念 profile8 CPE 维度: 9
正分支对象特征维度: 34
负分支对象特征维度: 34
正概念节点数: 12640, 正概念特征维度: 12, 正边数: 13824/173785
负概念节点数: 19473, 负概念特征维度: 12, 负边数: 13824/5907821
TensorBoard 日志将保存在: ../runs/car_weighted_bipartite_transformer_cpe_profile8_topk8_20260623-170533

--- 开始训练 weighted bipartite Transformer, topK=8 ---
topK=8, Epoch: 001, Loss: 1.4518, Train Acc: 0.6786, Val Acc: 0.6580, Test Acc: 0.6945


topK=8, Epoch: 002, Loss: 1.2462, Train Acc: 0.7056, Val Acc: 0.6899, Test Acc: 0.7147
topK=8, Epoch: 003, Loss: 1.0687, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


topK=8, Epoch: 004, Loss: 0.9225, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147
topK=8, Epoch: 005, Loss: 0.8004, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


topK=8, Epoch: 006, Loss: 0.7162, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147
topK=8, Epoch: 007, Loss: 0.6553, Train Acc: 0.6998, Val Acc: 0.6870, Test Acc: 0.7147


topK=8, Epoch: 008, Loss: 0.6048, Train Acc: 0.7181, Val Acc: 0.6957, Test Acc: 0.7233
topK=8, Epoch: 009, Loss: 0.5612, Train Acc: 0.8079, Val Acc: 0.7971, Test Acc: 0.8300
topK=8, Epoch: 010, Loss: 0.5018, Train Acc: 0.8948, Val Acc: 0.9159, Test Acc: 0.9164


...

topK=32, Epoch: 146, Loss: 0.0025, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


topK=32, Epoch: 147, Loss: 0.0022, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


topK=32, Epoch: 148, Loss: 0.0021, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


topK=32, Epoch: 149, Loss: 0.0023, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000


topK=32, Epoch: 150, Loss: 0.0024, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 1.0000
--- topK=32 训练完成 ---
topK=32 最终测试集准确率: 1.0000

--- 实验汇总 ---
{'topk': 8, 'final_train_acc': 1.0, 'final_val_acc': 1.0, 'final_test_acc': 1.0, 'log_dir': '../runs/car_weighted_bipartite_transformer_cpe_profile8_topk8_20260623-170533'}
{'topk': 32, 'final_train_acc': 1.0, 'final_val_acc': 1.0, 'final_test_acc': 1.0, 'log_dir': '../runs/car_weighted_bipartite_transformer_cpe_profile8_topk32_20260623-170551'}
